In [ ]:
import pandas as pd

movies = pd.read_csv("C:\\Users\\hp\\Documents\\Projects\\movie_recommender\\data\\tmdb_5000_movies.csv")
movies.head(2)
credits = pd.read_csv("C:\\Users\\hp\\Documents\\Projects\\movie_recommender\\data\\tmdb_5000_credits.csv")
credits.head(2)
movies = movies.merge(credits, on='title')
movies.head(2)

,movie_id,title,cast,crew,budget,genres,homepage,id,keywords,original_language,...,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,vote_average,vote_count
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,...,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,7.2,11800
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,...,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",6.9,4500


In [ ]:
#selecting important columns
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [ ]:
movies.dropna(inplace=True)
#convert json columns (extracting names)
import ast
def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name'])
    return L
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

In [7]:
#extract top 3 cast
def convert_cast(text):
    L=[]
    counter =0 
    for i in ast.literal_eval(text):
        if counter<3:
            L.append(i["name"])
            counter+=1
    return L
movies["cast"] = movies["cast"].apply(convert_cast)


In [9]:
#extracting directors
def fetch_director(text):
    L=[] 
    for i in ast.literal_eval(text):
        if i["job"]=="Director":
            L.append(i["name"])
    return L
movies["crew"] = movies["crew"].apply(fetch_director)

In [ ]:
#convert overview to list
movies["overview"] = movies["overview"].apply(lambda x:x.split())   

#combining all the features into one
movies["tags"] = movies["overview"] + movies["genres"] + movies["keywords"] + movies["cast"] + movies["crew"]

new_df = movies[["movie_id","title","tags"]]
new_df.head(2)

,movie_id,title,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."


In [11]:
#converting list to string
new_df["tags"] = new_df["tags"].apply(lambda x: " ".join(x))
new_df["tags"]= new_df["tags"].apply(lambda x:x.lower())
new_df.head(2)

C:\Users\hp\AppData\Local\Temp\ipykernel_16676\2181003139.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df["tags"] = new_df["tags"].apply(lambda x: " ".join(x))
C:\Users\hp\AppData\Local\Temp\ipykernel_16676\2181003139.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df["tags"]= new_df["tags"].apply(lambda x:x.lower())


,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."


In [12]:
#vectorization
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000,stop_words="english")
vectors = cv.fit_transform(new_df["tags"]).toarray()
vectors.shape

(4806, 5000)

In [13]:
#cosine similarity
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)

In [14]:
#recommendation function
def recommend(movie):
    movie_index = new_df[new_df["title"]==movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6]
    
    for i in movies_list:
        print(new_df.iloc[i[0]].title)

In [15]:
#testing the function
recommend("The Dark Knight Rises")

The Dark Knight
Batman Begins
Batman
Batman
Batman Returns
